# Initial Dimension Load – Car Workshop

Run **once** after `create_tables.sql`. Builds the 7 dimension tables and writes
each as a **parquet snapshot** to the external volume on ADLS:
`/Volumes/car_workshop/dim/landing/<table_name>/` (overwrite – safe to re-run).

The generation logic lives in **`simulator/dims_generation.py`** (seed 42, shared
with the local Docker lab); this notebook only provides the Spark writer.

Prerequisite: the `car_workshop.dim.landing` external volume must exist
(`infra/create_external_adls.sql`, section 3). File -> `car_workshop.dim.*`
ingestion is a separate step (see `journal/todo/04-sheets-redesign.md`).

| table | rows |
|---|---|
| locations | 99 (one per city in CITIES) |
| employees | ~1,160 |
| customers | 50,000 |
| vehicles | 65,000 |
| products | ~930 (product names x 2-6 manufacturers) |
| services | 96 (= SERVICE_CATALOGUE) |
| suppliers | 500 |

In [0]:
%python
%pip install faker

In [0]:
%python
import sys
import os

# Add both parent directory and Classes directory to path for imports
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
classes_dir = os.path.join(parent_dir, 'Classes')

for path in [parent_dir, classes_dir]:
    if path not in sys.path:
        sys.path.insert(0, path)

from Classes.dim_generator import generate_dims
from Classes.table_schemas import TABLE_SCHEMAS, schema_to_ddl

CATALOG = 'car_workshop'
LANDING = f'/Volumes/{CATALOG}/bronze/landing'


def save_dim(pdf, table_name):
    df = spark.createDataFrame(pdf, schema=schema_to_ddl(TABLE_SCHEMAS[table_name]))
    path = f'{LANDING}/{table_name}'
    df.write.mode('overwrite').parquet(path)
    print(f'  {table_name}: {len(pdf):,} rows -> {path}')


stats = generate_dims(save_dim)  # scale=1.0 -> the full deterministic dataset

In [0]:
%python
for table in ['locations', 'employees', 'customers', 'dim_vehicles',
              'products', 'services', 'suppliers']:
    n = spark.read.parquet(f'{LANDING}/{table}').count()
    print(f'{table}: {n:,} rows')

In [0]:
%python
parent_dir

In [0]:
%python
classes_dir